# ETL - Clasificador de Reclamos

Pipeline de extraccion, transformacion y carga del dataset de quejas de PROFECO.

**Pasos:**
1. Carga del dataset original (Quejas.xlsx)
2. Diagnostico de nulos
3. Limpieza selectiva (solo eliminar filas sin target)
4. Agrupacion del target en 8 macro-categorias
5. Construccion del campo de texto sintetico
6. Exportacion del dataset limpio

In [1]:
import pandas as pd
import numpy as np

print("Librerias cargadas")

Librerias cargadas


C:\Users\MARCO\AppData\Local\Temp\ipykernel_20492\4229914337.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


## 1. Carga del Dataset Original

In [2]:
df = pd.read_excel("Quejas.xlsx")

print(f"Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"\nColumnas: {df.columns.tolist()}")

ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.

In [ ]:
df.head(3)

## 2. Diagnostico de Nulos

In [ ]:
# Reemplazar guiones '-' por NaN real
df = df.replace("-", np.nan)

print(f"{'COLUMNA':<35} {'NULOS':>8}  {'%':>6}")
print("-" * 55)
for col in df.columns:
    total_missing = df[col].isnull().sum()
    pct = total_missing / len(df) * 100
    flag = " <<<" if pct > 5 else ""
    print(f"{col:<35} {total_missing:>8,}  {pct:>5.1f}%{flag}")

## 3. Limpieza Selectiva

**Estrategia:** Solo eliminar filas donde el **target** (`MOTIVO_RECLAMACION`) o `TIPO_RECLAMACION` son nulos,
ya que estas columnas son indispensables. El resto de nulos se imputan.

Esto conserva ~60,600 filas en vez de las ~27,400 que quedaban con `dropna()` global.

In [ ]:
# Eliminar filas sin target o sin tipo de reclamacion
antes = len(df)
df = df.dropna(subset=["MOTIVO_RECLAMACION", "TIPO_RECLAMACION"])
print(f"Filas eliminadas por target nulo: {antes - len(df):,}")
print(f"Filas conservadas: {len(df):,} ({len(df)/antes*100:.1f}%)")

In [ ]:
# Imputar nulos en columnas categoricas con "Desconocido"
cat_cols = ["SECTOR", "PROCEDIMIENTO", "BIEN O SERV", "MEDIO INGRESO",
            "TIPO PROD", "MODALIDAD COMPRA", "MODALIDAD PAGO", "PROB ESPECIAL"]

for col in cat_cols:
    nulos = df[col].isnull().sum()
    if nulos > 0:
        df[col] = df[col].fillna("Desconocido")
        print(f"  {col}: {nulos} nulos imputados")

print(f"\nNulos restantes en categoricas: {df[cat_cols].isnull().sum().sum()}")

## 4. Agrupacion del Target en Macro-Categorias

El target original (`MOTIVO_RECLAMACION`) tiene ~137 categorias, muchas con menos de 10 registros.
Agrupamos en **8 macro-categorias** por afinidad tematica para que el modelo pueda generalizar.

In [ ]:
CATEGORY_MAP = {
    # --- 1. Entrega del producto o servicio ---
    "Negativa a la entrega del producto o servicio": "Entrega del producto o servicio",
    "Negativa a la entrega del servicio": "Entrega del producto o servicio",
    "Incumplimiento de plazos para la entrega del prod. o serv.": "Entrega del producto o servicio",
    "No se entrego producto o servicio": "Entrega del producto o servicio",
    "No se entreg\u00f3 producto o servicio": "Entrega del producto o servicio",
    "Cancelacion de vuelo": "Entrega del producto o servicio",
    "Cancelaci\u00f3n de vuelo": "Entrega del producto o servicio",
    "Vuelo demorado": "Entrega del producto o servicio",
    "Retraso de vuelo": "Entrega del producto o servicio",
    "Cambio de itinerario": "Entrega del producto o servicio",
    "Sobreventa": "Entrega del producto o servicio",
    "sobreventa": "Entrega del producto o servicio",
    "P\u00e9rdida de vuelo": "Entrega del producto o servicio",
    "Negativa al servicio": "Entrega del producto o servicio",
    "Da\u00f1os durante el proceso de entrega": "Entrega del producto o servicio",
    "Suspensi\u00f3n de la provisi\u00f3n del servicio": "Entrega del producto o servicio",
    "Deficiencia del servicio ": "Entrega del producto o servicio",
    "Deficiencia del servicio": "Entrega del producto o servicio",
    "Negativa a la venta de un bien o a la prestaci\u00f3n de un servicio": "Entrega del producto o servicio",
    "Negativa a la cancelaci\u00f3n del servicio y devoluci\u00f3n": "Entrega del producto o servicio",
    "Negativa a cancelar el servicio": "Entrega del producto o servicio",

    # --- 2. Garantias y reparaciones ---
    "Negativa a hacer efectiva la garant\u00eda": "Garantias y reparaciones",
    "Defectos de fabricaci\u00f3n": "Garantias y reparaciones",
    "Defectos de fabricacion": "Garantias y reparaciones",
    "Deficiencia en la reparaci\u00f3n": "Garantias y reparaciones",
    "Garant\u00eda sin requisitos de ley": "Garantias y reparaciones",
    "Garant\u00eda-Info incomp.": "Garantias y reparaciones",
    "Garant\u00eda-Info no clara": "Garantias y reparaciones",
    "Garant\u00eda-Info otros idiomas": "Garantias y reparaciones",
    "No re\u00fane requisitos de ley": "Garantias y reparaciones",
    "No re\u00fane requisitos de ley-Inf ilegible": "Garantias y reparaciones",
    "Alteraci\u00f3n de instrumentos de medici\u00f3n": "Garantias y reparaciones",
    "Alteraci\u00f3n del contenido de productos envasados": "Garantias y reparaciones",

    # --- 3. Cambios y devoluciones ---
    "Negativa a cambio o devoluci\u00f3n": "Cambios y devoluciones",
    "Negativa a cambio o devolucion": "Cambios y devoluciones",
    "Producto o servicio equivocado": "Cambios y devoluciones",
    "Negativa a devoluci\u00f3n del precio pagado": "Cambios y devoluciones",
    "NEGATIVA A DEVOLUCION DEL PRECIO PAGADO": "Cambios y devoluciones",
    "Negativa a bonificaci\u00f3n": "Cambios y devoluciones",
    "Negativa a bonificaci\u00f3n por cambio de producto": "Cambios y devoluciones",
    "Negativa a efectuar la bonificaci\u00f3n": "Cambios y devoluciones",
    "Negativa a cambios": "Cambios y devoluciones",
    "Negativa a cambio": "Cambios y devoluciones",
    "Solicitud de cambio de producto": "Cambios y devoluciones",
    "Solicitud de la devoluci\u00f3n de pago": "Cambios y devoluciones",
    "Solicitud de ajuste de servicio y devoluci\u00f3n": "Cambios y devoluciones",
    "Pol\u00edtica de cambios y devoluciones-Info no clara": "Cambios y devoluciones",
    "Pol\u00edtica de cambios y devoluciones": "Cambios y devoluciones",
    "Da\u00f1o o extravio de equipaje": "Cambios y devoluciones",
    "P\u00e9rdida de equipaje": "Cambios y devoluciones",
    "Aver\u00eda de equipaje": "Cambios y devoluciones",

    # --- 4. Cobros indebidos ---
    "Negativa a corregir errores de cobro": "Cobros indebidos",
    "Error de c\u00e1lculo": "Cobros indebidos",
    "Cobro de cuota extraordinaria": "Cobros indebidos",
    "Producto o servicio no solicitado o autorizado": "Cobros indebidos",
    "Cobro indebido": "Cobros indebidos",
    "Cobro de equipaje": "Cobros indebidos",
    "cobro de equipaje": "Cobros indebidos",
    "Cobro excesivo en cambio de vuelo": "Cobros indebidos",
    "Cargos autom\u00e1ticos": "Cobros indebidos",
    "Cargos autom\u00e1ticos informaci\u00f3n": "Cobros indebidos",
    "Cargos autom\u00e1ticos Neg.": "Cobros indebidos",
    "Por alteraci\u00f3n de precio o tarifa m\u00e1ximo u oficial": "Cobros indebidos",
    "Modificaci\u00f3n del precio convenido o presupuestado": "Cobros indebidos",
    "No respeta precios anunciados": "Cobros indebidos",
    "No respeta descuentos": "Cobros indebidos",
    "No respeta promociones anunciadas": "Cobros indebidos",
    "No respeta garant\u00eda de precio bajo": "Cobros indebidos",
    "Aumento de tarifa": "Cobros indebidos",
    "Precio o tarifa": "Cobros indebidos",
    "Precio o tarifa-Info no clara": "Cobros indebidos",
    "Precio o tarifa m\u00e1xima u oficial": "Cobros indebidos",
    "Intereses": "Cobros indebidos",
    "Intereses Inf": "Cobros indebidos",
    "Intereses-Informa": "Cobros indebidos",
    "Comisiones": "Cobros indebidos",
    "Comisiones Neg": "Cobros indebidos",
    "Estados de cuenta negativa": "Cobros indebidos",
    "Estados de cuenta-Informa": "Cobros indebidos",
    "Estados de cuenta": "Cobros indebidos",
    "Uso fraudulento": "Cobros indebidos",
    "Aplicaci\u00f3n incorrecta de forma de pago": "Cobros indebidos",
    "Problemas al efectuar el pago y acreditar la compra": "Cobros indebidos",
    "Pagos a capital": "Cobros indebidos",
    "Periodicidad de pagos": "Cobros indebidos",
    "Forma de pago": "Cobros indebidos",
    "Forma de pago Neg": "Cobros indebidos",
    "Medio de pago": "Cobros indebidos",
    "Medio de pago Neg.": "Cobros indebidos",
    "Descuentos y bonificaciones-Informa": "Cobros indebidos",
    "Condiciones de pago": "Cobros indebidos",
    "Pago menor al convenido por p\u00e9rdida": "Cobros indebidos",
    "Pago menor al convenido por deterioro": "Cobros indebidos",

    # --- 5. Depositos y reembolsos ---
    "Negativa a la devoluci\u00f3n de dep\u00f3sito": "Depositos y reembolsos",
    "Negativa a pago por deterioro del producto": "Depositos y reembolsos",
    "Negativa a pago por p\u00e9rdida del producto": "Depositos y reembolsos",
    "Negativa a pago por p\u00e9rdidas o deterioro a consecuencia del uso del producto": "Depositos y reembolsos",
    "Negativa a pagar costos adicionales incurridos por el consumidor": "Depositos y reembolsos",
    "Penalizaci\u00f3n por causa imputable al proveedor": "Depositos y reembolsos",
    "Penalizaciones": "Depositos y reembolsos",

    # --- 6. Contratos ---
    "Negativa a la rescisi\u00f3n del contrato": "Contratos",
    "Modificaci\u00f3n o rescisi\u00f3n sin aviso ni autorizaci\u00f3n": "Contratos",
    "Contrato de comercializaci\u00f3n de bienes no determinados o no determinables": "Contratos",
    "Plazos, cantidades, condiciones": "Contratos",
    "Contiene cl\u00e1usulas abusivas": "Contratos",
    "No respet\u00f3 acuerdo previo": "Contratos",
    "No cumple con acuerdo": "Contratos",
    "No cumple acuerdo": "Contratos",
    "Negativa a la renovaci\u00f3n": "Contratos",
    "Negativa a la entrega de comprobante": "Contratos",
    "No se entreg\u00f3 por escrito": "Contratos",
    "Comprobantes-Informa": "Contratos",
    "Constituci\u00f3n de grupos": "Contratos",
    "Incumplimiento en los plazos ": "Contratos",
    "Incumplimiento en los plazos": "Contratos",
    "Incumplimiento en la oferta": "Contratos",
    "Negativa a respetar la oferta": "Contratos",
    "Acreditaci\u00f3n de la propiedad": "Contratos",
    "Autorizaciones y licencias para efectuar la venta o proveer el servicio": "Contratos",
    "No registrado ante Profeco": "Contratos",
    "Condicionamiento de venta": "Contratos",

    # --- 7. Informacion y publicidad ---
    "Publicidad enga\u00f1osa": "Informacion y publicidad",
    "Descripci\u00f3n del producto o servicio": "Informacion y publicidad",
    "Descripci\u00f3n del producto o servicio-Info no clara": "Informacion y publicidad",
    "Descripci\u00f3n del producto o servicio-Info incomp.": "Informacion y publicidad",
    "Descripci\u00f3n del producto o servicio-Info otros idiomas": "Informacion y publicidad",
    "Responsabilidad del proveedor por actos de sus dependientes": "Informacion y publicidad",
    "Informaci\u00f3n insuficiente": "Informacion y publicidad",
    "Etiquetado y empaque-Info incomp.": "Informacion y publicidad",
    "Instructivo": "Informacion y publicidad",
    "Env\u00edo de informaci\u00f3n, promociones u ofertas no solicitadas por tel\u00e9fono": "Informacion y publicidad",
    "Cesi\u00f3n, utilizaci\u00f3n o transmisi\u00f3n de informaci\u00f3n del consumidor a terceros": "Informacion y publicidad",
    "Aviso en medio masivo para hacer efectivo un cobro o cumplimiento de un contrato": "Informacion y publicidad",

    # --- 8. Otros ---
    "Malos tratos o represalias": "Otros",
    "Pr\u00e1cticas coercitivas o desleales de venta": "Otros",
    "Pr\u00e1cticas discriminatorias": "Otros",
    "Pr\u00e1cticas discriminatorias a discapacitados": "Otros",
    "Negativa a facturaci\u00f3n": "Otros",
    "Condiciones de seguridad": "Otros",
    "Condiciones de seguridad-Ausentes": "Otros",
    "Condiciones de seguridad-Info incomp.": "Otros",
    "Indeterminable": "Otros",
    "indeterminable": "Otros",
    "DESBLOQUEO": "Otros",
}

# Aplicar mapeo; categorias no mapeadas van a "Otros"
df["CATEGORIA"] = df["MOTIVO_RECLAMACION"].map(CATEGORY_MAP).fillna("Otros")

print("Distribucion de macro-categorias:")
print("-" * 50)
dist = df["CATEGORIA"].value_counts()
for cat, count in dist.items():
    pct = count / len(df) * 100
    barra = "#" * int(pct)
    print(f"  {cat:<35} {count:>6,}  ({pct:>5.1f}%) {barra}")

## 5. Construccion del Texto Sintetico

Concatenamos todas las columnas categoricas descriptivas para crear un campo de texto
que el modelo TF-IDF + XGBoost pueda procesar.

In [ ]:
text_columns = ["TIPO_RECLAMACION", "SECTOR", "BIEN O SERV", "TIPO PROD",
                "MODALIDAD COMPRA", "MODALIDAD PAGO", "MEDIO INGRESO",
                "PROCEDIMIENTO", "PROB ESPECIAL"]

df["texto"] = df[text_columns].fillna("").astype(str).agg(" ".join, axis=1).str.lower().str.strip()

print("Campo 'texto' creado.")
print(f"Longitud promedio: {df['texto'].str.len().mean():.0f} caracteres")
print(f"\nEjemplos:")
for i in range(3):
    print(f"  [{df['CATEGORIA'].iloc[i]}] -> {df['texto'].iloc[i][:120]}...")

## 6. Exportacion del Dataset Limpio

In [ ]:
# Seleccionar solo las columnas necesarias para el modelo
df_export = df[["texto", "CATEGORIA"]].copy()

print(f"Dataset final: {len(df_export):,} filas")
print(f"Columnas: {df_export.columns.tolist()}")
print(f"Categorias: {df_export['CATEGORIA'].nunique()}")

df_export.to_csv("clasificador-reclamos-backend/data/quejas_limpias.csv", index=False, encoding="utf-8-sig")
print("\nArchivo guardado: clasificador-reclamos-backend/data/quejas_limpias.csv")

In [ ]:
df_export.head(10)